# Sparrow: Arrow arrays in modern C++20

This notebook runs entirely in your browser: [xeus-cpp](https://github.com/compiler-research/xeus-cpp) interprets C++20 compiled to WebAssembly,
with [sparrow](https://github.com/sparrow-org/sparrow) pre-installed.

The demo notebooks: [sparrow.ipynb](sparrow.ipynb) (arrays), [sparrow-ipc.ipynb](sparrow-ipc.ipynb) (Arrow IPC with pyarrow), [sparrow-compute.ipynb](sparrow-compute.ipynb) (lazy compute) and [rockfinch.ipynb](rockfinch.ipynb) (C++ extension module for Python).

Run the cells from top to bottom (`Shift+Enter`). A C++ name can only be defined once per kernel: restart the kernel before re-running the notebook.

In [ ]:
#pragma clang diagnostic ignored "-W#warnings"
// Must match the compile definitions of the WebAssembly sparrow package
#define SPARROW_USE_LARGE_INT_PLACEHOLDERS
#include <format>
#include <iostream>
#include <string>
#include <vector>

#include <sparrow.hpp>
#include <sparrow/record_batch.hpp>
#include <sparrow/config/sparrow_version.hpp>

namespace sp = sparrow;

std::cout << "sparrow " << sp::SPARROW_VERSION_MAJOR << "." << sp::SPARROW_VERSION_MINOR << "." << sp::SPARROW_VERSION_PATCH << std::endl;

## Low level primitives: `nullable`

Represents a possibly-missing value, with an API similar to `std::optional`.

In [ ]:
sp::nullable<int> n1;
sp::nullable<int> n2 = 42;
sp::nullable<int> n3 = sp::make_nullable(7, false);
std::cout << std::boolalpha;
std::cout << "n1.has_value(): " << n1.has_value() << std::endl;
std::cout << "n2.has_value(): " << n2.has_value() << ", n2.value(): " << n2.value() << std::endl;
std::cout << "n3.has_value(): " << n3.has_value() << ", n3.get(): " << n3.get() << std::endl;

## Typed arrays

One array class per Arrow layout, all with almost the same API: value semantics, iterators, mutability.

In [ ]:
std::vector<int32_t> values = {1, 2, 3, 4, 5};
std::vector<bool> validity = {true, true, false, true, true};
sp::primitive_array<int32_t> ar{values, validity, "my_array"};
std::cout << std::format("{}", ar) << std::endl;

In [ ]:
std::cout << "name: "               << ar.name().value()   << std::endl;
std::cout << "size: "               << ar.size()           << std::endl;
std::cout << "front: "              << ar.front().value()  << std::endl;
std::cout << "back: "               << ar.back().value()   << std::endl;
std::cout << "ar[2].has_value(): "  << ar[2].has_value()   << std::endl;
std::cout << "ar[2].value_or(42): " << ar[2].value_or(42)  << std::endl;

In [ ]:
for (const auto& nullable_value : ar)
{
    std::cout << (nullable_value.has_value() ? std::to_string(nullable_value.value()) : "null") << ", ";
}
std::cout << std::endl;
for (const auto& value : ar.values())
{
    std::cout << value << ", ";
}

In [ ]:
ar[3] = 0;
ar.push_back(sp::make_nullable(111, true));
ar.insert(ar.cbegin() + 1, sp::make_nullable(222, true));
std::cout << std::format("{}", ar) << std::endl;

## Arrow C Data Interface and untyped arrays

Typed arrays can hand over their `ArrowArray` / `ArrowSchema` structures; `sp::array` wraps any Arrow data behind a type-erased API.

In [ ]:
auto [arrow_array, arrow_schema] = sp::extract_arrow_structures(std::move(ar));
std::cout << "format: " << arrow_schema.format << ", length: " << arrow_array.length
          << ", null_count: " << arrow_array.null_count << std::endl;

In [ ]:
sp::array untyped(std::move(arrow_array), std::move(arrow_schema));
std::cout << "name: "                    << untyped.name().value()  << std::endl;
std::cout << "size: "                    << untyped.size()          << std::endl;
std::cout << "untyped[3].has_value(): "  << untyped[3].has_value()  << std::endl;
untyped.visit([]<class T>(const T& typed_ar)
{
    if constexpr (sp::is_primitive_array_v<T>)
    {
        std::cout << "visited a primitive array: " << std::format("{}", typed_ar) << std::endl;
    }
});

## Record batch

A schema plus equal-length columns.

In [ ]:
sp::record_batch rb(
    {{"id", sp::array(sp::primitive_array<int32_t>{1, 2, 3, 4})},
     {"value", sp::array(sp::primitive_array<double>{1.5, 2.5, 3.5, 4.5})},
     {"name", sp::array(sp::string_array(std::vector<std::string>{"alpha", "beta", "gamma", "delta"}))}});
std::cout << "columns: " << rb.nb_columns() << ", rows: " << rb.nb_rows() << std::endl;

In [ ]:
void print_batch(const sp::record_batch& batch)
{
    for (std::size_t i = 0; i < batch.nb_columns(); ++i)
    {
        std::cout << std::format("{}", batch.get_column(i)) << std::endl;
    }
}

In [ ]:
print_batch(rb);